# 03 - Data Structures and Algorithms (Python)

This notebook builds the fault-stack / action-queue / CAN-ID-lookup example from `concept.md` idiomatically in Python, then runs a small Big-O demo comparing linear search against a dict lookup. Read `concept.md` first if you haven't.

## Stack: Recent Controller Faults

Python's built-in `list` already supports stack operations directly: `.append()` pushes onto the end, `.pop()` removes and returns from the end. We treat "the end of the list" as "the top of the stack."

In [1]:
fault_stack = []

fault_stack.append("CAN timeout: device 12")
fault_stack.append("Brownout detected")
fault_stack.append("CAN timeout: device 7")

print("Most recent fault first:")
while fault_stack:
    print(" -", fault_stack.pop())

Most recent fault first:
 - CAN timeout: device 7
 - Brownout detected
 - CAN timeout: device 12


Notice the last fault pushed ("CAN timeout: device 7") is the first one popped — Last In, First Out. This is exactly what you want for a driver-station fault display: whatever just went wrong is the first thing shown.

## Queue: Autonomous Action Sequence

A plain `list` *can* act as a queue with `.pop(0)`, but that's O(n) every time (removing from the front means shifting every remaining element over). `collections.deque` is built specifically to make both ends fast: `.append()` to enqueue, `.popleft()` to dequeue.

In [2]:
from collections import deque

action_queue = deque()
action_queue.append("drive forward")
action_queue.append("intake")
action_queue.append("shoot")
action_queue.append("drive back")

print("Executing in queued order:")
while action_queue:
    print(" -", action_queue.popleft())

Executing in queued order:
 - drive forward
 - intake
 - shoot
 - drive back


Here the *first* action queued ("drive forward") is the *first* one executed — First In, First Out. That's the whole point: autonomous steps have to run in the order they were planned, not reversed.

## Hashmap: CAN ID to Device Name

Python's `dict` is a hashmap. Looking up a value by key doesn't require scanning every entry — Python computes a hash of the key and goes almost straight to the matching slot.

In [3]:
can_id_to_name = {
    1: "Front Left Drive",
    2: "Front Right Drive",
    3: "Back Left Drive",
    4: "Back Right Drive",
    12: "Intake Roller",
}

print(can_id_to_name[12])
print(can_id_to_name.get(99, "<unknown device>"))  # .get() avoids a crash on a missing key

Intake Roller
<unknown device>


`can_id_to_name[12]` goes directly to "Intake Roller" regardless of how many other devices are in the map. `.get(key, default)` is the safe way to look up a key that might not exist, without an extra `if key in can_id_to_name` check first.

## Tree: The Same CAN ID Table, Sorted

A `dict` doesn't keep entries in any particular order guaranteed by key, because iterating over `can_id_to_name` gives them back in insertion order, not sorted by key. A **binary search tree (BST)** keeps entries sorted by key while still supporting fast lookup: every node holds a key/value pair plus a link to a left child (smaller keys) and a right child (larger keys). Python's standard library doesn't ship a BST the way it ships `dict`, so we build one by hand.

In [ ]:
class DeviceTree:
    class _Node:
        __slots__ = ("can_id", "name", "left", "right")

        def __init__(self, can_id, name):
            self.can_id = can_id
            self.name = name
            self.left = None
            self.right = None

    def __init__(self):
        self._root = None

    def insert(self, can_id, name):
        self._root = self._insert(self._root, can_id, name)

    def _insert(self, node, can_id, name):
        if node is None:
            return DeviceTree._Node(can_id, name)
        if can_id < node.can_id:
            node.left = self._insert(node.left, can_id, name)
        elif can_id > node.can_id:
            node.right = self._insert(node.right, can_id, name)
        else:
            node.name = name  # Same ID seen again -- update rather than duplicate.
        return node

    def find(self, can_id):
        node = self._root
        while node is not None:
            if can_id == node.can_id:
                return node.name
            node = node.left if can_id < node.can_id else node.right
        return None

    def in_order(self):
        yield from self._in_order(self._root)

    def _in_order(self, node):
        if node is None:
            return
        yield from self._in_order(node.left)
        yield (node.can_id, node.name)
        yield from self._in_order(node.right)

Only these five devices go into the tree, inserted in arbitrary (not sorted) order — deliberately **not** the 10,000-entry table used in the Big-O demo below. Inserting keys that are already sorted into a plain BST like this one builds a completely lopsided tree (every node with only a right child), which defeats O(log n) entirely; a real tree implementation would rebalance itself instead, which is out of scope here.

In [ ]:
device_tree = DeviceTree()
device_tree.insert(4, "Back Right Drive")
device_tree.insert(1, "Front Left Drive")
device_tree.insert(12, "Intake Roller")
device_tree.insert(2, "Front Right Drive")
device_tree.insert(3, "Back Left Drive")

print(device_tree.find(12))
found = device_tree.find(99)
print(found if found is not None else "<unknown device>")

print("\nAll devices, sorted by CAN ID (in-order tree walk):")
for can_id, name in device_tree.in_order():
    print(f" - {can_id}: {name}")

Notice the in-order walk prints every device sorted by CAN ID (1, 2, 3, 4, 12) even though we inserted them out of order (4, 1, 12, 2, 3) — that sorted-for-free property is exactly what a `dict` can't give you.

## Big-O in Practice: Linear Search vs. Binary Search vs. Dict Lookup

To make the O(n) vs. O(log n) vs. O(1) difference concrete instead of abstract, we'll build a much bigger table of (id, name) pairs and count how many comparisons a linear scan and a binary search each need to find a given ID, at three different positions: near the front, near the back, and missing entirely.

In [4]:
def linear_search_comparisons(entries, target_id):
    """Scan a list of (id, name) tuples from the front. Return the number
    of comparisons made before finding target_id (or exhausting the list)."""
    comparisons = 0
    for entry_id, _name in entries:
        comparisons += 1
        if entry_id == target_id:
            return comparisons
    return comparisons


def binary_search_comparisons(sorted_entries, target_id):
    """Binary search a list of (id, name) tuples that's already sorted by id.
    Return the number of comparisons made. Gives wrong answers, not just slow
    ones, if sorted_entries isn't actually sorted."""
    comparisons = 0
    low, high = 0, len(sorted_entries) - 1
    while low <= high:
        comparisons += 1
        mid = (low + high) // 2
        mid_id = sorted_entries[mid][0]
        if mid_id == target_id:
            return comparisons
        elif mid_id < target_id:
            low = mid + 1
        else:
            high = mid - 1
    return comparisons


# A big table: 10,000 fake CAN IDs, in order -- already sorted by id, which
# is exactly what binary_search_comparisons requires.
big_table = [(i, f"device-{i}") for i in range(10_000)]
big_map = dict(big_table)

for target in [5, 5_000, 9_999]:
    linear_comparisons = linear_search_comparisons(big_table, target)
    binary_comparisons = binary_search_comparisons(big_table, target)
    print(f"id={target:>5} -> linear: {linear_comparisons:>5} comparisons, binary search: {binary_comparisons:>2} comparisons")

print()
print(f"dict lookup for id=5:     {big_map[5]}")
print(f"dict lookup for id=9999:  {big_map[9999]}")
print("(a dict lookup does roughly the same amount of work no matter where the entry is)")

linear search for id=    5:     6 comparisons
linear search for id= 5000:  5001 comparisons
linear search for id= 9999: 10000 comparisons

dict lookup for id=5:     device-5
dict lookup for id=9999:  device-9999
(a dict lookup does roughly the same amount of work no matter where the entry is)


The comparison count for linear search grows with the target's position — looking up ID `9999` takes about 2,000x more comparisons than looking up ID `5` in this table. Binary search barely grows at all: roughly log₂(n), a dozen or so comparisons whether the target is at position 5,000 or 9,999. The dict lookups above do essentially the same amount of work either way. That growing-with-position behavior *is* what O(n) means in practice, the barely-growing behavior *is* what O(log n) means, and the flat behavior *is* what O(1) means.

One thing you may notice: for `target = 5`, the linear scan can actually take *fewer* comparisons than binary search on that one specific call. That's not a contradiction — a linear scan's best case (an early match) is cheap no matter how big the data gets. Big-O describes the worst case *as data keeps growing*, not a promise that the asymptotically-better algorithm wins every single call. Binary search's guarantee is that it never gets much worse than ~log₂(n) regardless of where the target sits; linear search's worst case keeps getting worse as the table grows.

## Try It Yourself

No solutions are provided — these are meant to be worked through on your own or with a mentor or another student.

1. Add a `peek()`-style check to the fault stack example that looks at the most recent fault *without* removing it (hint: list indexing).
2. Modify `linear_search_comparisons` to also work for a target that isn't in the table at all, and confirm it returns the full length of `entries`.
3. Time (with `import time; time.perf_counter()`) an actual dict lookup versus an actual `linear_search_comparisons` call for `target = 9_999` on `big_table`/`big_map`, over many repetitions, and see if the wall-clock gap matches what the comparison counts predicted.
4. Add a `remove(can_id)` method to `DeviceTree`. (Hint: this is the hardest of the four — removing a node with two children means you need to pick a replacement, usually the smallest node in its right subtree, without breaking the BST ordering.)

In [5]:
# Your code here


## Resources

- [Python `collections.deque` docs](https://docs.python.org/3/library/collections.html#collections.deque) - the real implementation behind the queue example above.
- [Big-O Cheat Sheet](https://www.bigocheatsheet.com/) - time/space complexity for common data structures and algorithms.
- [GeeksforGeeks: Binary Search Tree](https://www.geeksforgeeks.org/binary-search-tree-data-structure/) - a deeper look at BST operations, including the delete/rebalancing cases this notebook doesn't cover.
- [WPILib `SequentialCommandGroup`](https://docs.wpilib.org/en/stable/docs/software/commandbased/command-groups.html) - the real queue-like structure behind chained autonomous actions on the robot.